In [0]:
# Filename: src/dlt_pipelines/streaming_pipeline.py
import dlt
from pyspark.sql.functions import col, to_timestamp

# --- SILVER LAYER: Cleaning & Deduplication [cite: 57, 127] ---
@dlt.table(
    name="events_cleaned",
    comment="Cleaned and deduplicated e-commerce events"
)
@dlt.expect_or_drop("valid_timestamp", "event_time IS NOT NULL") # DQ Constraint [cite: 43]
@dlt.expect_or_drop("valid_price", "price > 0") # DQ Constraint [cite: 43]
def events_cleaned():
    return (dlt.read_stream("events_raw") # Reads from Bronze
            .withColumn("event_time", to_timestamp(col("event_time")))
            .dropDuplicates(["user_id", "event_time", "product_id"]))

# --- GOLD LAYER: Aggregated Metrics [cite: 58, 128] ---
@dlt.table(
    name="customer_metrics",
    comment="Business Deliverable 1: Customer Journey Analysis"
)
def customer_metrics():
    return (dlt.read("events_cleaned")
            .groupBy("user_id")
            .agg({"product_id": "count", "price": "sum"})
            .withColumnRenamed("count(product_id)", "total_interactions")
            .withColumnRenamed("sum(price)", "total_spend"))